In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260905_123159_live"
folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260911_141429_live1"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
snapshots

,ts,trade_latency,depth_latency,exchange_latency,symbol,mid,mid_tick,microprice,microprice_dev,spread,...,quote_churn,micro_signal,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1789136071644,0,40,0,PEPEUSDT,0.000003,350,0.000003,-2.363970e-09,1.000000e-08,...,0.0,-0.000676,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0
1,1789136071744,0,40,0,PEPEUSDT,0.000003,350,0.000003,-2.363913e-09,1.000000e-08,...,0.0,-0.000676,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0
2,1789136071844,0,43,0,PEPEUSDT,0.000003,350,0.000003,-2.362275e-09,1.000000e-08,...,0.0,-0.000676,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0
3,1789136072044,0,40,43,PEPEUSDT,0.000003,350,0.000003,-2.362244e-09,1.000000e-08,...,0.0,-0.000676,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0
4,1789136072144,0,45,43,PEPEUSDT,0.000003,350,0.000003,-2.363893e-09,1.000000e-08,...,0.0,-0.000676,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2673,1789136458244,39,40,42,PEPEUSDT,0.000004,351,0.000004,-1.138251e-09,1.000000e-08,...,0.0,-0.000325,0.000004,0.0,0.000004,0.0,NaN,NaN,NaN,NaN
2674,1789136458444,39,41,42,PEPEUSDT,0.000004,351,0.000004,-1.110938e-09,1.000000e-08,...,0.0,-0.000317,0.000004,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2675,1789136458544,39,41,42,PEPEUSDT,0.000004,351,0.000004,-1.137710e-09,1.000000e-08,...,0.0,-0.000325,0.000004,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2676,1789136458644,39,40,42,PEPEUSDT,0.000004,351,0.000004,-1.137710e-09,1.000000e-08,...,0.0,-0.000325,0.000004,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
"""
Key Research Questions

This project is designed to investigate:

Which regimes favor passive liquidity provision? medium frequency trends (1000ms rolling window)

"""

# STEP 1 — Load raw data
df = snapshots
df["ts"] = pd.to_datetime(df["ts"], unit="ms")
df = df.set_index("ts")

# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

"""
2. Choose regime window (critical design choice)
Start simple:
"""

feature_cols = [
    "spread",
    "volatility",
    "order_imbalance",
    "trade_imbalance",
    "microprice_dev"
    # "quote_churn", # use only market features, not strategy features
]

WINDOW = "1s"   # later try 2s, 5s

regime_df = pd.DataFrame()

regime_df["spread"] = df["spread"].rolling(WINDOW).mean()
regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()
regime_df["microprice_dev"] = (df["microprice_dev"]).rolling(WINDOW).mean()
# regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()

regime_df = regime_df.dropna()
regime_df

,spread,volatility,order_imbalance,trade_imbalance,microprice_dev
ts,,,,,
2026-09-11 14:14:31.844,1.000000e-08,0.0,-0.472677,0.000000,-2.363386e-09
2026-09-11 14:14:32.044,1.000000e-08,0.0,-0.472620,0.000000,-2.363101e-09
2026-09-11 14:14:32.144,1.000000e-08,0.0,-0.472652,0.000000,-2.363259e-09
2026-09-11 14:14:32.244,1.000000e-08,0.0,-0.472673,0.000000,-2.363365e-09
2026-09-11 14:14:32.344,1.000000e-08,0.0,-0.472689,0.000000,-2.363444e-09
...,...,...,...,...,...
2026-09-11 14:20:58.244,1.000000e-08,0.0,-0.408557,0.609022,-2.042787e-09
2026-09-11 14:20:58.444,1.000000e-08,0.0,-0.362173,0.619311,-1.810864e-09
2026-09-11 14:20:58.544,1.000000e-08,0.0,-0.316457,0.629600,-1.582287e-09


In [3]:
# STEP 3 — Train regime model
scaler = StandardScaler()

X = regime_df[feature_cols].values
X_scaled = scaler.fit_transform(X)

n_regimes = 3  # start small: 2–5 max

model = GaussianMixture(
    n_components=n_regimes,
    covariance_type="full",
    random_state=42
)

regime_df["regime"] = model.fit_predict(X_scaled)

In [4]:
eval_df = df.copy()

times = eval_df.index          # DatetimeIndex
mid = eval_df["mid"].values

horizon_ms = 1000
HORIZON = pd.Timedelta(milliseconds=horizon_ms)

future_return = np.full(len(df), np.nan)
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df)):

    target_time = times[i] + HORIZON

    # first observation at or after t + 1000ms
    j = times.searchsorted(target_time)

    if j >= len(df):
        continue

    p0 = mid[i]
    p1 = mid[j]

    # future window [i, j]
    window = mid[i:j+1]

    # Need at least 2 observations
    if len(window) < 2:
        continue

    # 1. Future return
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility over next 1000ms
    returns = np.diff(window) / window[:-1]
    future_volatility[i] = np.std(returns)

    # 3. Future direction
    # If result ≈ +1
    # almost always up moves after this regime
    # strong bullish bias
    # If result ≈ -1
    # almost always down moves after this regime
    # bearish bias
    # If result ≈ 0
    # no directional bias
    # pure noise / mean reversion / stable
    future_direction[i] = np.sign(p1 - p0)

eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction

eval_df = eval_df.dropna(subset=[ "future_return", "future_volatility", "future_direction"])
eval_df

,trade_latency,depth_latency,exchange_latency,symbol,mid,mid_tick,microprice,microprice_dev,spread,best_bid,...,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms,future_return,future_volatility,future_direction
ts,,,,,,,,,,,,,,,,,,,,,
2026-09-11 14:14:31.644,0,40,0,PEPEUSDT,0.000003,350,0.000003,-2.363970e-09,1.000000e-08,0.000003,...,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.0,0.0,0.0
2026-09-11 14:14:31.744,0,40,0,PEPEUSDT,0.000003,350,0.000003,-2.363913e-09,1.000000e-08,0.000003,...,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.0,0.0,0.0
2026-09-11 14:14:31.844,0,43,0,PEPEUSDT,0.000003,350,0.000003,-2.362275e-09,1.000000e-08,0.000003,...,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.0,0.0,0.0
2026-09-11 14:14:32.044,0,40,43,PEPEUSDT,0.000003,350,0.000003,-2.362244e-09,1.000000e-08,0.000003,...,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.0,0.0,0.0
2026-09-11 14:14:32.144,0,45,43,PEPEUSDT,0.000003,350,0.000003,-2.363893e-09,1.000000e-08,0.000003,...,0.0,0.000003,0.0,0.000003,0.0,0.000003,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-11 14:20:57.144,41,41,42,PEPEUSDT,0.000004,351,0.000004,-2.966327e-09,1.000000e-08,0.000003,...,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN,0.0,0.0,0.0
2026-09-11 14:20:57.244,41,41,42,PEPEUSDT,0.000004,351,0.000004,-2.998995e-09,1.000000e-08,0.000003,...,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN,0.0,0.0,0.0
2026-09-11 14:20:57.444,41,40,42,PEPEUSDT,0.000004,351,0.000004,-2.966327e-09,1.000000e-08,0.000003,...,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN,0.0,0.0,0.0


In [5]:
# STEP 5 — ALIGN BOTH DATASETS

# Now regime + outcome are aligned.

final = regime_df.merge(
    eval_df[["future_return", "future_volatility", "future_direction"]],
    left_index=True,
    right_index=True,
    how="inner"
)

# STEP 6 — ANALYZE REGIMES
regime_outcomes = final.groupby("regime").agg({
    "future_return": "mean",
    "future_volatility": "mean",
    "future_direction": "mean"
})

z = final.copy()

for col in feature_cols:
    z[col] = (z[col] - z[col].mean()) / z[col].std()

regime_profile = (
    z.groupby("regime")[feature_cols]
    .mean()
    .round(2)
)

full_profile = pd.DataFrame(regime_profile.join(regime_outcomes))
full_profile

,spread,volatility,order_imbalance,trade_imbalance,microprice_dev,future_return,future_volatility,future_direction
regime,,,,,,,,
0,-0.05,-0.35,0.16,0.00,0.16,0.000072,0.000067,0.024989
1,-0.08,2.81,0.09,-0.11,0.09,0.000109,0.000185,0.037671
2,0.76,-0.35,-2.19,0.19,-2.19,-0.001064,0.000370,-0.372093


In [6]:
def export_gmm(model_name, gmm, scaler, horizon_ms, feature_cols, regime_labels):
    K = gmm.n_components

    means = gmm.means_

    covs = gmm.covariances_
    precisions = gmm.precisions_  # inverse covariance (what you want)

    log_weights = np.log(gmm.weights_)

    # log determinant of covariance
    log_det = np.array([
        np.log(np.linalg.det(covs[k]))
        for k in range(K)
    ])

    artifact = {
        # GMM
        "means": means.tolist(),
        "cov_inv": precisions.tolist(),
        "log_det_cov": log_det.tolist(),
        "log_weights": log_weights.tolist(),

        # scaler (CRITICAL)
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),

        # metadata
        "model_name": model_name,
        "target": "detect_regime",
        "n_regimes": K,
        "horizon_ms": horizon_ms,
        "feature_cols": feature_cols,
        "regime_labels": [
            regime_labels[i] for i in range(K)
        ]
    }

    with open(f"data/{model_name}.json", "w") as f:
        json.dump(artifact, f)

    print(f"['data/{model_name}.json']")

In [ ]:
regime_labels = {
    0: "low_vol",
    1: "high_vol",
    2: "directional",
}

export_gmm(model_name="regime_model_pepe",
           gmm=model,
           scaler=scaler,
           horizon_ms=horizon_ms,
           feature_cols=feature_cols,
           regime_labels=regime_labels)

['data/regime_model_pepe.json']
